# 02 · What does a pretrained network see in our helmets?

**ResNet50** is a well known image network. It was trained on **ImageNet**, a public set of 1.2 million photos in 1,000 classes (dog, car, toaster, …). We use it as it is, without training anything.

ImageNet has no class for "construction helmet" and no color classes. The closest classes are `crash_helmet` (motorcycle helmet) and `football_helmet`.

**Question:** which label does ResNet50 give to each of our helmets, and does that label already separate the colors?

The answer explains why, in notebook 04, we replace the last layer of the network.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps

REPO = Path.cwd().parent                      # the repository folder (notebooks run from notebooks/)
HELMETS = REPO / "data/helmets"
OUTPUTS = REPO / "outputs"; OUTPUTS.mkdir(exist_ok=True)
SIZE = 224                                    # ResNet50 input size in pixels
COLORS = ["green", "orange", "red", "transparent", "yellow"]

meta = pd.read_csv(HELMETS / "helmets_metadata.csv")
# "All image in the Dataset" holds exact copies (same MD5) of "Cartella", so we use only "Cartella"
photos = meta[meta.dataset_folder == "Cartella DATASET Algoritmo convegno"].reset_index(drop=True)
assert photos.md5.is_unique and set(photos.helmet_color) == set(COLORS)

def load(path):
    im = ImageOps.exif_transpose(Image.open(HELMETS / path)).convert("RGB")  # apply camera rotation; drop PNG alpha
    return np.asarray(im.resize((SIZE, SIZE), Image.BILINEAR), dtype=np.uint8)


from keras.applications.resnet50 import ResNet50, preprocess_input, decode_predictions

X = np.stack([load(p) for p in photos.relative_path])
model = ResNet50(weights="imagenet")          # full network with its 1,000 ImageNet classes
prob = model.predict(preprocess_input(X.astype("float32")), batch_size=32, verbose=0)
top5 = decode_predictions(prob, top=5)        # the 5 most likely ImageNet classes for each photo
photos["top1"] = [t[0][1] for t in top5]
photos["helmet_in_top5"] = [any("helmet" in name for _, name, _ in t) for t in top5]
print(prob.shape)

(174, 1000)


## 1. Most likely label (top-1) for each color

In [2]:
table = pd.crosstab(photos.top1, photos.helmet_color).reindex(columns=COLORS)
table = table.loc[table.sum(axis=1).sort_values(ascending=False).index]
table.to_csv(OUTPUTS / "02_top1_per_color.csv")
table.head(10)

helmet_color,green,orange,red,transparent,yellow
top1,,,,,
crash_helmet,8,13,14,4,7
perfume,2,0,10,4,0
football_helmet,0,1,1,1,12
pencil_sharpener,3,4,4,0,0
toaster,0,0,0,0,9
vacuum,0,6,0,0,1
vase,3,0,1,1,1
piggy_bank,1,4,1,0,0
spotlight,5,0,0,0,0


In [3]:
photos.groupby("helmet_color").agg(
    photos=("top1", "size"),
    most_common_label=("top1", lambda s: s.mode()[0]),
    pct_most_common=("top1", lambda s: round(100 * (s == s.mode()[0]).mean(), 1)),
    pct_helmet_in_top5=("helmet_in_top5", lambda s: round(100 * s.mean(), 1)))

,photos,most_common_label,pct_most_common,pct_helmet_in_top5
helmet_color,,,,
green,28,crash_helmet,28.6,64.3
orange,31,crash_helmet,41.9,87.1
red,33,crash_helmet,42.4,63.6
transparent,30,bubble,13.3,40.0
yellow,52,football_helmet,23.1,57.7


## 2. What we learned
- **Without retraining, ResNet50 does not classify by color.** It gives object names, not colors.
- The same label, `crash_helmet`, appears for all 5 colors. So the last ImageNet layer is not useful for this task.
- What ResNet50 computes **before** its last layer is still useful: a list of 2,048 numbers that describes the image (the *feature vector*). In notebook 04 we keep that part and train a new last layer with 5 colors. This is called **transfer learning**.